In [ ]:
import sys 
sys.path.append('..')

import pickle
import pandas as pd

from scripts.global_vars import (
    DEVICE, 
    N_TOPICS, 
    SAMPLE_SIZE, 
    COLUMN_NAME,
    REDUCE_TOPICS, 
    USE_TRAIN_CHECKPOINT
)

from scripts.pipeline.bert_pipeline import bertopic_pipeline, inference_pipeline
from scripts.pipeline.visualization import plot_topics_wordcloud, plot_topics_cluster

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

---
### **STAGE 1. PREPROCESSING**
---

In [ ]:
df = pd.read_csv("../data/corpus/topic_dataset.csv")
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)
sample_df

In [ ]:
corpus = sample_df[COLUMN_NAME].tolist()
corpus

---
### **STAGE 2. MODEL EXPERIMENTS**
---

In [ ]:
if USE_TRAIN_CHECKPOINT:
    topic_model = pickle.load(open(f"../data/models/bertopic_model_{COLUMN_NAME}.pkl", "rb"))
    topics, probs = pickle.load(open(f"../data/inference/bertopic_inference_{COLUMN_NAME}.pkl", "rb")).values()

else:
    topic_model = bertopic_pipeline(
        corpus, 
        device=DEVICE,
        reduce_topics=REDUCE_TOPICS, 
        nr_topics=N_TOPICS, 
        save=f"../data/models/bertopic_model_{COLUMN_NAME}.pkl"
    )
    
    topics, probs = inference_pipeline(
        topic_model, 
        corpus, 
        save=f"../data/inference/bertopic_inference_{COLUMN_NAME}.pkl"
    )

In [ ]:
sample_df['topic'] = topics
sample_df['probability'] = probs.max(axis=1)
sample_df.loc[sample_df["topic"] == -1, "probability"] = 0.0

sample_df

In [ ]:
topics = topic_model.get_topics()
topic_info = topic_model.get_topic_info()

topic_info

---
### **STAGE 3. VISUALIZE ALMOST ANYTHING!**
---

In [ ]:
plot_topics_cluster(sample_df, topic_model, 20)

In [ ]:
plot_topics_wordcloud(topics, topic_model)

In [ ]:
# sample_df_filtered = sample_df[sample_df['topic'] != -1]

# topics_filtered = sample_df_filtered['topic'].tolist()
# corpus_filtered = sample_df_filtered[COLUMN_NAME].tolist()
# timestamps = [i.isoformat() for i in sample_df_filtered['publish_date'].tolist()]

# topics_over_time = topic_model.topics_over_time(
#     corpus_filtered,
#     timestamps,
#     topics=topics_filtered,
#     nr_bins=40
# )

# topic_model.visualize_topics_over_time(
#     topics_over_time, 
#     title="Topic Evolution Over Time",
#     top_n_topics=40,
#     width=1600,
#     height=700
# )

In [ ]:
topic_model.visualize_heatmap(
    n_clusters=6, 
    width=900, 
    height=900,
    title="Topic Similarity Heatmap"
)

In [ ]:
topic_model.visualize_barchart(
    title="Top Words per Topic",
    top_n_topics=20, 
    n_words=15,
    height=300,
    width=300
)

In [ ]:
topic_model.visualize_hierarchy(
    title="Topic Hierarchy - Hierarchical Clustering of Topics",
    width=1000,
    height=1000
)